In [ ]:
import __main__
import os
import sys

sys.modules["__main__"].__file__ = "model.py"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [ ]:
import os
import time
import json
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from datasets import load_dataset, interleave_datasets
from transformers import AutoTokenizer
import huggingface_hub
import wandb

import warnings

warnings.filterwarnings("ignore")

In [3]:
torch.cuda.memory._record_memory_history(max_entries=100000)

In [4]:
%%writefile dataset.py
# dataset.py

import torch
from torch.utils.data import Dataset, DataLoader

class IsolatedPackedDataset(Dataset):
    def __init__(self, tokenized_docs, block_size=512, bos_token_id=1):
        """
        Args:
            tokenized_docs: List of lists, where each sublist contains token IDs for one document.
            block_size: The maximum context window size of your model.
            bos_token_id: The ID corresponding to your Beginning-of-Stream (<s>) token.
        """
        self.block_size = block_size
        self.bos_token_id = bos_token_id
        self.packed_samples = []
        
        current_packet = []
        
        for doc in tokenized_docs:
            if len(doc) == 0:
                continue
            if doc[0] != self.bos_token_id:
                doc = [self.bos_token_id] + doc
                
            if len(doc) > self.block_size:
                doc = doc[:self.block_size]
                
            if len(current_packet) + len(doc) <= self.block_size:
                current_packet.extend(doc)
            else:
                remainder = self.block_size - len(current_packet)
                if remainder > 0:
                    current_packet.extend([0] * remainder)
                
                self.packed_samples.append(current_packet)
                current_packet = list(doc)
                
        if current_packet:
            remainder = self.block_size - len(current_packet)
            current_packet.extend([0] * remainder)
            self.packed_samples.append(current_packet)

    def __len__(self):
        return len(self.packed_samples)

    def __getitem__(self, idx):
        tokens = torch.tensor(self.packed_samples[idx], dtype=torch.long)
        
        # Shift tokens by 1 step to generate target labels for auto-regressive prediction
        x = tokens[:-1]
        y = tokens[1:]
        
        position_ids = torch.zeros_like(x)
        current_pos = 0
        
        for i in range(len(x)):
            if x[i] == self.bos_token_id and i > 0:
                current_pos = 0
            position_ids[i] = current_pos
            current_pos += 1
            
        bos_indices = (x == self.bos_token_id).nonzero(as_tuple=True)[0].tolist()
        bos_indices.append(len(x)) 
        seq_len = len(x)
        mask = torch.full((seq_len, seq_len), float("-inf"))
        
        for start_idx, end_idx in zip(bos_indices[:-1], bos_indices[1:]):
            sub_mask = torch.tril(torch.ones(end_idx - start_idx, end_idx - start_idx))
            mask[start_idx:end_idx, start_idx:end_idx] = sub_mask.masked_fill(sub_mask == 0, float("-inf")).masked_fill(sub_mask == 1, 0.0)
            
        return x, y, position_ids, mask

Writing dataset.py


In [ ]:
%%writefile model.py


import math
import torch
import torch.nn as nn
from dataclasses import dataclass
from transformers import PretrainedConfig, PreTrainedModel
from transformers.generation import GenerationMixin
from transformers.modeling_outputs import CausalLMOutputWithPast


@dataclass
class Config(PretrainedConfig):
    model_type = "amartya-slm"
    
    def __init__(
        self,
        vocab_size=12288,
        block_size=512,
        n_layer=8,
        n_head=8,
        n_kv_head=4,
        n_embd=512,
        use_rope=True,
        use_rmsnorm=True,
        use_swiglu=True,
        tie_word_embeddings=True,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_kv_head = kwargs.get("n_kv_head", 4)
        self.n_embd = n_embd
        self.use_rope = use_rope
        self.use_rmsnorm = use_rmsnorm
        self.use_swiglu = use_swiglu


    
    
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight

class ModularNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.use_rmsnorm:
            self.norm = RMSNorm(config.n_embd)
        else:
            self.norm = nn.LayerNorm(config.n_embd)
            
    def forward(self, x):
        return self.norm(x)
    

    
class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden_dim = int(2 * (4 * config.n_embd) / 3)
        self.w1 = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w2 = nn.Linear(config.n_embd, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, config.n_embd, bias=False)
        self.w3.is_residual_proj = True

    def forward(self, x):
        
        return self.w3(torch.nn.functional.silu(self.w1(x)) * self.w2(x))

class ModularMLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        if config.use_swiglu:
            self.mlp = SwiGLU(config)
        else:
            self.mlp = nn.Sequential(
                nn.Linear(config.n_embd, 4 * config.n_embd),
                nn.GELU(),
                nn.Linear(4 * config.n_embd, config.n_embd)
            )
            
    def forward(self, x):
        return self.mlp(x)
    
    
    
def precompute_freqs(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    
    freqs_cos = torch.cos(freqs)
    freqs_sin = torch.sin(freqs)
    return freqs_cos, freqs_sin

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(xq, xk, freqs_cos, freqs_sin):
    if freqs_cos.dim() == 2:
        freqs_cos = freqs_cos.unsqueeze(0).unsqueeze(2)
        freqs_sin = freqs_sin.unsqueeze(0).unsqueeze(2)
    elif freqs_cos.dim() == 3:
        freqs_cos = freqs_cos.unsqueeze(2)
        freqs_sin = freqs_sin.unsqueeze(2)

    freqs_cos = torch.cat([freqs_cos, freqs_cos], dim=-1)
    freqs_sin = torch.cat([freqs_sin, freqs_sin], dim=-1)

    xq_out = (xq * freqs_cos) + (rotate_half(xq) * freqs_sin)
    xk_out = (xk * freqs_cos) + (rotate_half(xk) * freqs_sin)

    return xq_out.type_as(xq), xk_out.type_as(xk)


class ModularAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.n_head % config.n_kv_head == 0
        
        self.n_head = config.n_head
        self.n_kv_head = config.n_kv_head
        self.n_embd = config.n_embd
        self.use_rope = config.use_rope
        self.head_dim = config.n_embd // config.n_head


        self.n_rep = self.n_head // self.n_kv_head 
        self.q_proj = nn.Linear(config.n_embd, self.n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.n_embd, self.n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.n_embd, self.n_kv_head * self.head_dim, bias=False)
        
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=False)
        self.c_proj.is_residual_proj = True
        
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))

    def forward(self, x, custom_mask=None, freqs_cos=None, freqs_sin=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim)
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim)
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim)

        if self.use_rope and freqs_cos is not None:
            q, k = apply_rotary_emb(q, k, freqs_cos, freqs_sin)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        k = torch.repeat_interleave(k, repeats=self.n_rep, dim=1)
        v = torch.repeat_interleave(v, repeats=self.n_rep, dim=1)
        
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        if torch.isnan(att).any():
            print("NaN in attention scores")
            raise RuntimeError("NaN attention")
            
        if custom_mask is not None:
            if custom_mask.dim() == 3:
                custom_mask = custom_mask.unsqueeze(1)
            att = att + custom_mask
            
        else:
            
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            
        att = torch.nn.functional.softmax(att, dim=-1)
        y = att @ v
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))    
    


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = ModularNorm(config)
        self.attn = ModularAttention(config)
        self.ln_2 = ModularNorm(config)
        self.mlp = ModularMLP(config)

    def forward(self, x, freqs_cos=None, freqs_sin=None, custom_mask=None):
        
        x = x + self.attn(self.ln_1(x), custom_mask=custom_mask, freqs_cos=freqs_cos, freqs_sin=freqs_sin)
        x = x + self.mlp(self.ln_2(x))
        return x


class SmallLanguageModel(PreTrainedModel, GenerationMixin):
    config_class = Config
    
    _tied_weights_keys = {"lm_head.weight": None}

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        
        if not config.use_rope:
            self.wpe = nn.Embedding(config.block_size, config.n_embd)
            
        self.blocks = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = ModularNorm(config)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        
        
        self.lm_head.weight = self.wte.weight

        if config.use_rope:
            freqs_cos, freqs_sin = precompute_freqs(config.n_embd // config.n_head, config.block_size * 2)
            self.register_buffer("freqs_cos", freqs_cos)
            self.register_buffer("freqs_sin", freqs_sin)
            
        self.post_init()

        
    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            if hasattr(module, 'is_residual_proj') and module.is_residual_proj:
                with torch.no_grad():
                    module.weight.mul_(1.0 / math.sqrt(2.0 * self.config.n_layer))
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        self.lm_head.weight = self.wte.weight

    def forward(self, idx, targets=None, custom_mask=None, pos_ids=None, attention_mask=None, return_dict=None, **kwargs):
        B, T = idx.size()
        x = self.wte(idx)

        if not self.config.use_rope:
            pos = pos_ids if pos_ids is not None else torch.arange(0, T, dtype=torch.long, device=idx.device)
            x = x + self.wpe(pos)

        if self.config.use_rope:
            if pos_ids is not None: 
                f_cos = self.freqs_cos[pos_ids] 
                f_sin = self.freqs_sin[pos_ids]
            else:
                f_cos = self.freqs_cos[:T]
                f_sin = self.freqs_sin[:T]
        else:
            f_cos, f_sin = None, None

        for block in self.blocks:
            x = block(x, freqs_cos=f_cos, freqs_sin=f_sin, custom_mask=custom_mask)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = torch.nn.functional.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        if return_dict:
            return CausalLMOutputWithPast(
                loss=loss,
                logits=logits,
                past_key_values=None,
                hidden_states=None,
                attentions=None
            )

        return logits, loss

    def prepare_inputs_for_generation(self, input_ids, attention_mask=None, **kwargs):
        if input_ids.shape[1] > self.config.block_size:
            input_ids = input_ids[:, -self.config.block_size:]
        
        return {
            "idx": input_ids,
            "custom_mask": None,
            "pos_ids": None,
            "attention_mask": attention_mask
        }



Writing model.py


In [ ]:
from model import *
from dataset import *
from transformers import AutoConfig, AutoModelForCausalLM

Config.register_for_auto_class()
SmallLanguageModel.register_for_auto_class("AutoModelForCausalLM")

## 1. ENVIRONMENT & WANDB TRACKING CONFIGURATION


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing pipeline on hardware target: {device}")

from kaggle_secrets import UserSecretsClient

secret_client = UserSecretsClient()

WANDB_API_KEY = secret_client.get_secret("WANDB_API_KEY")
HF_WRITE_TOKEN = secret_client.get_secret("HF_TOKEN")


wandb.login(key=WANDB_API_KEY)
huggingface_hub.login(token=HF_WRITE_TOKEN)

Executing pipeline on hardware target: cuda


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pandeyamartya (pandeyamartya-mmmut) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
wandb.init(
    project="SLMa-training",
    name="slm-94M",
    config={
        "learning_rate": 1e-3,
        "max_steps": 48000,
        "vocab_size": 24576,
        "block_size": 768,
        "n_layer": 12,
        "n_head": 12,
        "n_embd": 768,
        "architecture": "Llama-Style GQA + SwiGLU + RoPE",
    },
)

wandb: setting up run oai0ozeu
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260603_000905-oai0ozeu
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run slm-94M
wandb: ⭐️ View project at https://wandb.ai/pandeyamartya-mmmut/SLMa-training
wandb: 🚀 View run at https://wandb.ai/pandeyamartya-mmmut/SLMa-training/runs/oai0ozeu


In [ ]:
# download tookenizer
TOKENIZER_REPO = "amartya-pandey/slm-tokenizer-24k"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_REPO)

tokenizer_config.json:   0%|          | 0.00/307 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 2. MODEL CORE CONFIGURATION


In [ ]:
config = Config(
    vocab_size=len(tokenizer),
    block_size=768,
    n_layer=12,
    n_head=12,
    n_kv_head=4,
    n_embd=768,
    use_rope=True,
    use_rmsnorm=True,
    use_swiglu=True,
)
model = SmallLanguageModel(config).to(device)

In [11]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 94,391,040
Trainable Parameters: 94,391,040


In [ ]:
SUBSETS = [
    "auto_math_text",
    "khanacademy",
    "openstax",
    "stanford",
    "stories",
    "web_samples_v1",
    "web_samples_v2",
    "wikihow",
]

PROBS = [0.15, 0.20, 0.20, 0.10, 0.02, 0.15, 0.15, 0.03]

print("Loading...")
streams = [
    load_dataset(
        "HuggingFaceTB/cosmopedia",
        name=s,
        split="train",
        streaming=True,
    )
    for s in SUBSETS
]

mixed_dataset = interleave_datasets(streams, probabilities=PROBS, seed=42)
print("Complete!!")

Loading...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/139 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/118 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Complete!!


In [ ]:
start = time.perf_counter()
print("Fetching and tokenizing text records over the network...")
tokenized_docs = []
max_documents_to_buffer = 3_000_000

for idx, row in enumerate(mixed_dataset):
    if idx >= max_documents_to_buffer:
        break
    ids = tokenizer.encode(row["text"])
    tokenized_docs.append(ids)
print("Complete!!")
end = time.perf_counter()
execution_time = end - start

print(f"Training took {execution_time:.0f} seconds to run.")

Fetching and tokenizing text records over the network...
Complete!!
Training took 418 seconds to run.


In [ ]:
print("Assembling token strings into isolated, packed block-diagonal matrices...")
dataset = IsolatedPackedDataset(
    tokenized_docs, block_size=config.block_size, bos_token_id=tokenizer.bos_token_id
)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

Assembling token strings into isolated, packed block-diagonal matrices...


In [ ]:
learning_rate = 1e-3


def get_optimizer_params(model, weight_decay=0.01):
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}

    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]

    optim_groups = [
        {"params": decay_params, "weight_decay": weight_decay},
        {"params": nodecay_params, "weight_decay": 0.0},
    ]
    return optim_groups


optim_groups = get_optimizer_params(model, weight_decay=0.01)
optimizer = optim.AdamW(optim_groups, lr=1e-3, betas=(0.9, 0.95), eps=1e-8)
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)
scaler = torch.amp.GradScaler("cuda")

In [ ]:
import math


def get_lr(step, max_steps, warmup_steps, max_lr=1e-3, min_lr=1e-4):
    """Calculates learning rate with linear warmup followed by cosine decay."""
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    if step > max_steps:
        return min_lr

    decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))  # Ranges from 1.0 down to 0.0

    return min_lr + coeff * (max_lr - min_lr)

In [18]:
model.train()

SmallLanguageModel(
  (wte): Embedding(24576, 768)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (ln_1): ModularNorm(
        (norm): RMSNorm()
      )
      (attn): ModularAttention(
        (q_proj): Linear(in_features=768, out_features=768, bias=False)
        (k_proj): Linear(in_features=768, out_features=256, bias=False)
        (v_proj): Linear(in_features=768, out_features=256, bias=False)
        (c_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (ln_2): ModularNorm(
        (norm): RMSNorm()
      )
      (mlp): ModularMLP(
        (mlp): SwiGLU(
          (w1): Linear(in_features=768, out_features=2048, bias=False)
          (w2): Linear(in_features=768, out_features=2048, bias=False)
          (w3): Linear(in_features=2048, out_features=768, bias=False)
        )
      )
    )
  )
  (ln_f): ModularNorm(
    (norm): RMSNorm()
  )
  (lm_head): Linear(in_features=768, out_features=24576, bias=False)
)

In [19]:
max_steps = 48000
warmup_steps = 900
max_learning_rate = 1e-3
min_learning_rate = 1e-4

accumulation_steps = 4

In [ ]:
# 5. THE RUNNING ENGINE EXECUTOR
start = time.perf_counter()

step = 0
print(
    f"\n Core Engine ready. Starting training loops (Grad Accumulation: {accumulation_steps})..."
)

optimizer.zero_grad()

while step < max_steps:
    for x, y, pos_ids, mask in dataloader:
        if step >= max_steps:
            break

        # Dynamically calculate and update learning rate for the current step
        current_lr = get_lr(
            step=step,
            max_steps=max_steps,
            warmup_steps=warmup_steps,
            max_lr=max_learning_rate,
            min_lr=min_learning_rate,
        )
        for param_group in optimizer.param_groups:
            param_group["lr"] = current_lr

        x, y = x.to(device), y.to(device)
        mask = mask.to(device)
        pos_ids = pos_ids.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits, loss = model(x, custom_mask=mask, pos_ids=pos_ids, targets=y)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        if step % 200 == 0:
            wandb.log(
                {
                    "step": step,
                    "train_loss": loss.item() * accumulation_steps,
                    "learning_rate": current_lr,  # Track your scheduler trajectory
                }
            )

        if step % 1000 == 0:
            print(
                f"Step {step:05d} | Loss: {loss.item() * accumulation_steps:.4f} | LR: {current_lr:.6f}"
            )

        step += 1

end = time.perf_counter()
execution_time = end - start

print(f"Training took {execution_time:.0f} seconds to run.")
wandb.finish()


 Core Engine ready. Starting training loops (Grad Accumulation: 4)...
Step 00000 | Loss: 10.1303 | LR: 0.000001
Step 01000 | Loss: 4.2192 | LR: 0.001000
Step 02000 | Loss: 3.8236 | LR: 0.000999
Step 03000 | Loss: 3.3109 | LR: 0.000996
Step 04000 | Loss: 2.8678 | LR: 0.000990
Step 05000 | Loss: 2.9855 | LR: 0.000983
Step 06000 | Loss: 3.1909 | LR: 0.000974
Step 07000 | Loss: 2.3655 | LR: 0.000963
Step 08000 | Loss: 2.3791 | LR: 0.000950
Step 09000 | Loss: 2.3755 | LR: 0.000936
Step 10000 | Loss: 2.2611 | LR: 0.000920
Step 11000 | Loss: 2.5779 | LR: 0.000902
Step 12000 | Loss: 2.9014 | LR: 0.000882
Step 13000 | Loss: 2.7984 | LR: 0.000861
Step 14000 | Loss: 2.1776 | LR: 0.000839
Step 15000 | Loss: 2.5657 | LR: 0.000815
Step 16000 | Loss: 2.4334 | LR: 0.000790
Step 17000 | Loss: 2.3454 | LR: 0.000765
Step 18000 | Loss: 2.1783 | LR: 0.000738
Step 19000 | Loss: 2.2325 | LR: 0.000710
Step 20000 | Loss: 2.1481 | LR: 0.000682
Step 21000 | Loss: 2.4482 | LR: 0.000653
Step 22000 | Loss: 2.1924 

wandb: updating run metadata


Training took 41413 seconds to run.


wandb: uploading summary, console lines 58-58
wandb: 
wandb: Run history:
wandb: learning_rate ▄▅█████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁
wandb:          step ▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇█
wandb:    train_loss █▃▄▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▁▁▁▁▁▂▁▁▁▁
wandb: 
wandb: Run summary:
wandb: learning_rate 0.0001
wandb:          step 47800
wandb:    train_loss 1.35498
wandb: 
wandb: 🚀 View run slm-94M at: https://wandb.ai/pandeyamartya-mmmut/SLMa-training/runs/oai0ozeu
wandb: ⭐️ View project at: https://wandb.ai/pandeyamartya-mmmut/SLMa-training
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260603_000905-oai0ozeu/logs


In [ ]:
# Define our definitive local distribution directory and target cloud repo name
FINAL_EXPORT_DIR = "./slm_unified_pretrained_model"
os.makedirs(FINAL_EXPORT_DIR, exist_ok=True)

CLOUD_REPO_NAME = "SLMa-94M"
print(f"Exporting model tensors and configuration to local path: {FINAL_EXPORT_DIR}")
model.save_pretrained(FINAL_EXPORT_DIR)

Exporting model tensors and configuration to local path: ./slm_unified_pretrained_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [22]:
torch.cuda.memory._dump_snapshot("gpu_profile.pkl")

In [ ]:
try:
    model.push_to_hub(
        repo_id=CLOUD_REPO_NAME,
        commit_message="Prod grade",
    )
    print("-> Model tensors uploaded successfully!")


except Exception as e:
    print("Push failed! Check your token permissions or network routes.")
    print(f"Error Details: {e}")
    print(
        f"Note: Your artifacts are still saved safely in your local directory: {FINAL_EXPORT_DIR}"
    )

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

-> Model tensors uploaded successfully!
